# Graph Neural Networks — Exercises

8 exercises covering MPNN implementation, GCN propagation, WL expressiveness, GAT attention, GIN, over-smoothing diagnosis, graph pooling, and graph transformer positional encodings.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1-3 | Core mechanics |
| ★★ | 4-6 | Deeper theory |
| ★★★ | 7-8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
|-------|----------|
| GCN propagation matrix | 1 |
| MPNN message passing | 2 |
| Aggregation expressiveness | 3 |
| WL color refinement | 4 |
| GAT attention | 5 |
| Over-smoothing analysis | 6 |
| GIN expressiveness | 7 |
| Graph Transformer (LapPE) | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la
from collections import Counter
from itertools import permutations

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '='*len(title))
    print(title)
    print('='*len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond

def make_adj_matrix(n, edges):
    A = np.zeros((n, n))
    for u, v in edges:
        A[u,v] = 1; A[v,u] = 1
    return A

def make_adj_dict(n, edges):
    adj = {i: set() for i in range(n)}
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    return adj

print('Setup complete.')

---

## Exercise 1 ★ — GCN Propagation Matrix

Derive and verify properties of the GCN propagation matrix $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$.

**Graph:** Path graph $P_5$ with nodes $\{0,1,2,3,4\}$ and edges $\{(0,1),(1,2),(2,3),(3,4)\}$.

(a) Build the adjacency matrix $A$ and compute $\tilde{A} = A + I$, $\tilde{D}$.

(b) Compute $\hat{A}$ and verify all eigenvalues lie in $(-1, 1]$.

(c) Compute the stationary distribution $\boldsymbol{\pi}$ of the random walk on $\tilde{A}$ and verify $\hat{A}^{50}$ converges to a rank-1 matrix with rows proportional to $\boldsymbol{\pi}$.

(d) For initial features $X = I_5$, compute $H^{[3]} = \hat{A}^3 X$ and measure the Dirichlet energy $E(H^{[3]}) = \operatorname{tr}(H^{[3]\top} L H^{[3]})$.

In [ ]:
# Exercise 1: Your Solution
n = 5
edges_P5 = [(0,1),(1,2),(2,3),(3,4)]
A = make_adj_matrix(n, edges_P5)

# (a) Compute A_tilde and D_tilde
A_tilde = None  # YOUR CODE HERE
D_tilde = None  # YOUR CODE HERE

# (b) Compute A_hat
A_hat = None    # YOUR CODE HERE
eigvals = None  # YOUR CODE HERE

# (c) Stationary distribution
pi = None       # YOUR CODE HERE

# (d) Dirichlet energy
L = np.diag(A.sum(axis=1)) - A
X = np.eye(n)
H3 = None       # YOUR CODE HERE
energy = None   # YOUR CODE HERE

print('A_tilde:', A_tilde)
print('A_hat:', A_hat)
print('Eigenvalues:', eigvals)
print('pi:', pi)
print('H3:', H3)
print('Energy:', energy)

In [ ]:
# Exercise 1: Solution
n = 5
edges_P5 = [(0,1),(1,2),(2,3),(3,4)]
A = make_adj_matrix(n, edges_P5)

# (a)
A_tilde = A + np.eye(n)
d_tilde = A_tilde.sum(axis=1)
D_tilde = np.diag(d_tilde)

# (b)
D_inv_sqrt = np.diag(1.0 / np.sqrt(d_tilde))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
eigvals = np.sort(np.linalg.eigvalsh(A_hat))[::-1]

# (c) Stationary distribution
pi = d_tilde / d_tilde.sum()
A_hat_50 = np.linalg.matrix_power(A_hat, 50)

# (d)
L = np.diag(A.sum(axis=1)) - A
X = np.eye(n)
H3 = np.linalg.matrix_power(A_hat, 3) @ X
energy = np.trace(H3.T @ L @ H3)

header('Exercise 1: GCN Propagation Matrix')
print(f'A_hat eigenvalues: {eigvals.round(4)}')
check_true('all eigenvalues in (-1,1]',
           eigvals.max() <= 1+1e-10 and eigvals.min() > -1-1e-10)
check_true('pi sums to 1', np.isclose(pi.sum(), 1.0))
check_true('A_hat^50 rows approx proportional to pi',
           np.allclose(A_hat_50[0] / A_hat_50[0].sum(),
                       A_hat_50[1] / A_hat_50[1].sum(), atol=1e-4))
check_true('Dirichlet energy is positive', energy > 0)
print(f'\nDirichlet energy E(H^[3]) = {energy:.6f}')
print(f'Stationary distribution: {pi.round(4)}')
print('\nTakeaway: A_hat^k smooths features; energy decays geometrically toward 0.')

---

## Exercise 2 ★ — MPNN Message Passing

Implement one layer of the MPNN framework from scratch with sum aggregation.

**Graph:** 6 nodes, edges $\{(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(0,3)\}$.

(a) Initialize node features $H^{[0]} = I_6$.

(b) For each node $v$, compute the message $\mathbf{m}_v = \sum_{u \in \mathcal{N}(v)} W_m \mathbf{h}_u$ where $W_m \in \mathbb{R}^{6 \times 4}$ is fixed (use seed 42).

(c) Update: $\mathbf{h}_v^{[1]} = \operatorname{ReLU}(W_u [\mathbf{h}_v \| \mathbf{m}_v])$ where $W_u \in \mathbb{R}^{10 \times 4}$.

(d) Verify permutation equivariance: permute nodes $[0,1,2,3,4,5] \to [3,0,5,2,4,1]$ and check the output permutes consistently.

In [ ]:
# Exercise 2: Your Solution
np.random.seed(42)
n2 = 6
edges2 = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(0,3)]
adj2 = make_adj_dict(n2, edges2)
H0_2 = np.eye(n2)
W_m = np.random.randn(6, 4) * 0.3
W_u = np.random.randn(10, 4) * 0.3

def mpnn_layer(H, adj, W_m, W_u):
    # YOUR CODE HERE
    pass

H1_2 = mpnn_layer(H0_2, adj2, W_m, W_u)
print('H1_2:', H1_2)

In [ ]:
# Exercise 2: Solution
np.random.seed(42)
n2 = 6
edges2 = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(0,3)]
adj2 = make_adj_dict(n2, edges2)
H0_2 = np.eye(n2)
W_m = np.random.randn(6, 4) * 0.3
W_u = np.random.randn(10, 4) * 0.3

def mpnn_layer(H, adj, W_m, W_u):
    H_new = np.zeros((len(adj), W_u.shape[1]))
    for v in adj:
        # (b) Message: sum over neighbors
        m_v = sum(H[u] @ W_m for u in adj[v]) if adj[v] else np.zeros(W_m.shape[1])
        # (c) Update: concat + ReLU
        concat = np.concatenate([H[v], m_v])
        H_new[v] = np.maximum(0, W_u.T @ concat)
    return H_new

H1_2 = mpnn_layer(H0_2, adj2, W_m, W_u)

# (d) Permutation equivariance
perm = [3,0,5,2,4,1]
P = np.eye(n2)[perm]
H0_perm = P @ H0_2
adj_perm = {perm[v]: {perm[u] for u in adj2[v]} for v in adj2}
H1_perm = mpnn_layer(H0_perm, adj_perm, W_m, W_u)

header('Exercise 2: MPNN Message Passing')
print(f'H1 shape: {H1_2.shape}')
check_true('all outputs non-negative (ReLU)', (H1_2 >= 0).all())
check_close('permutation equivariance: H1_perm == P @ H1',
            H1_perm, P @ H1_2, tol=1e-8)
print(f'\nTakeaway: MPNN is permutation equivariant because aggregation is over a set.')

---

## Exercise 3 ★ — Aggregation Expressiveness

Prove empirically that sum $\succ$ mean = max in discriminative power for multisets.

(a) For each pair of multisets below, compute sum, mean, and max. Identify which pairs each aggregator can distinguish.

| Pair | $\mathcal{M}_1$ | $\mathcal{M}_2$ |
|------|-----------------|------------------|
| A | $\{1, 1\}$ | $\{1, 1, 1\}$ |
| B | $\{1, 2\}$ | $\{2\}$ |
| C | $\{1, 1, 2\}$ | $\{1, 2, 2\}$ |

(b) Construct graph $G_1 = K_{1,4}$ (star with 4 leaves) and $G_2 = C_4$ (4-cycle) with initial node features all equal to 1. Apply one round of GCN (mean) and GIN (sum). Does GCN distinguish them? Does GIN?

(c) What structural difference between $G_1$ and $G_2$ allows sum (but not mean) to distinguish them?

In [ ]:
# Exercise 3: Your Solution
pairs = [
    ('A', [1.0, 1.0],       [1.0, 1.0, 1.0]),
    ('B', [1.0, 2.0],       [2.0]),
    ('C', [1.0, 1.0, 2.0],  [1.0, 2.0, 2.0]),
]

for name, m1, m2 in pairs:
    m1, m2 = np.array(m1), np.array(m2)
    # YOUR CODE HERE: compute sum, mean, max for m1 and m2
    # then determine if each aggregator distinguishes them
    pass

In [ ]:
# Exercise 3: Solution
pairs = [
    ('A', [1.0, 1.0],       [1.0, 1.0, 1.0]),
    ('B', [1.0, 2.0],       [2.0]),
    ('C', [1.0, 1.0, 2.0],  [1.0, 2.0, 2.0]),
]

header('Exercise 3: Aggregation Expressiveness')
print(f'{"Pair":<5} {"Sum?":<8} {"Mean?":<8} {"Max?"}')
print('-'*35)
all_ok = True
for name, m1, m2 in pairs:
    m1, m2 = np.array(m1), np.array(m2)
    s = not np.isclose(m1.sum(), m2.sum())
    me = not np.isclose(m1.mean(), m2.mean())
    mx = not np.isclose(m1.max(), m2.max())
    print(f'{name:<5} {str(s):<8} {str(me):<8} {str(mx)}')
    if name == 'A': all_ok = all_ok and (s and not me and not mx)
    if name == 'B': all_ok = all_ok and (s and not mx)
    if name == 'C': all_ok = all_ok and (s and me and not mx)

# (b) K_{1,4} vs C_4
# K_{1,4}: star, center has degree 4, leaves have degree 1
# C_4: cycle, all degree 2
# With all-ones features:
K14_center_mean_nbrs = 1.0  # All 4 neighbors have feature 1; mean = 1
C4_node_mean_nbrs = 1.0     # Both neighbors have feature 1; mean = 1

K14_center_sum_nbrs = 4.0   # 4 neighbors, sum = 4
C4_node_sum_nbrs = 2.0      # 2 neighbors, sum = 2

gcn_distinguishes = not np.isclose(K14_center_mean_nbrs, C4_node_mean_nbrs)
gin_distinguishes = not np.isclose(K14_center_sum_nbrs, C4_node_sum_nbrs)

print(f'\nK_{{1,4}} vs C_4 (uniform features):')
print(f'  GCN (mean): center_K14 mean={K14_center_mean_nbrs}, C4 mean={C4_node_mean_nbrs} -> distinguish: {gcn_distinguishes}')
print(f'  GIN (sum):  center_K14 sum={K14_center_sum_nbrs},  C4 sum={C4_node_sum_nbrs}   -> distinguish: {gin_distinguishes}')

check_true('sum distinguishes all 3 pairs', all_ok)
check_true('GIN distinguishes K_{1,4} vs C_4', gin_distinguishes)

print('\n(c) Structural difference: K_{1,4} center has 4 neighbors; C_4 nodes have 2.')
print('Sum is sensitive to neighborhood size; mean normalizes it away.')
print('\nTakeaway: sum aggregation is strictly more expressive than mean or max.')

---

## Exercise 4 ★★ — Weisfeiler-Leman Color Refinement

Implement 1-WL and use it to test pairs of graphs.

(a) Implement `wl_test(adj1, adj2)` that runs 1-WL color refinement on both graphs simultaneously and returns `True` if they are distinguishable, `False` if indistinguishable.

(b) Test on $C_6$ vs $C_3 \cup C_3$. What does WL conclude? Are they isomorphic?

(c) Test on $K_{1,3}$ (star) vs $P_4$ (path). Does WL correctly distinguish them?

(d) Test on two 4-regular graphs on 8 nodes. Do they look the same to 1-WL?

(e) Explain: what structural property does 1-WL fail to detect that causes it to confuse $C_6$ with $C_3 \cup C_3$?

In [ ]:
# Exercise 4: Your Solution
from collections import Counter

def wl_test(adj1, adj2, max_iter=10):
    """
    Run 1-WL on adj1 and adj2 (both as dicts {node: set_of_neighbors}).
    Returns True if WL distinguishes them, False if indistinguishable.
    """
    # YOUR CODE HERE
    pass

# Test graphs
C6_adj  = make_adj_dict(6, [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)])
C3C3_adj= make_adj_dict(6, [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3)])
K13_adj = make_adj_dict(4, [(0,1),(0,2),(0,3)])
P4_adj  = make_adj_dict(4, [(0,1),(1,2),(2,3)])

print('C6 vs C3+C3:', wl_test(C6_adj, C3C3_adj))
print('K13 vs P4:  ', wl_test(K13_adj, P4_adj))

In [ ]:
# Exercise 4: Solution
from collections import Counter

def wl_test(adj1, adj2, max_iter=10):
    def run_wl(adj):
        colors = {v: 0 for v in adj}
        for _ in range(max_iter):
            color_map = {}
            counter = [0]
            def get_c(key):
                if key not in color_map:
                    color_map[key] = counter[0]; counter[0] += 1
                return color_map[key]
            new_colors = {v: get_c((colors[v], tuple(sorted(colors[u] for u in adj[v])))) for v in adj}
            if new_colors == colors: break
            colors = new_colors
        return Counter(colors.values())

    h1 = run_wl(adj1); h2 = run_wl(adj2)
    return h1 != h2

C6_adj   = make_adj_dict(6, [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)])
C3C3_adj = make_adj_dict(6, [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3)])
K13_adj  = make_adj_dict(4, [(0,1),(0,2),(0,3)])
P4_adj   = make_adj_dict(4, [(0,1),(1,2),(2,3)])

# Two 3-regular graphs on 6 nodes: K_{3,3} vs prism
K33_adj  = make_adj_dict(6, [(0,3),(0,4),(0,5),(1,3),(1,4),(1,5),(2,3),(2,4),(2,5)])
prism_adj= make_adj_dict(6, [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3),(0,3),(1,4),(2,5)])

header('Exercise 4: Weisfeiler-Leman Test')
r1 = wl_test(C6_adj, C3C3_adj)
r2 = wl_test(K13_adj, P4_adj)
r3 = wl_test(K33_adj, prism_adj)

print(f'C6 vs C3+C3 (should be False — indistinguishable): {r1}')
print(f'K_{{1,3}} vs P4 (should be True — distinguishable):   {r2}')
print(f'K_{{3,3}} vs Prism (both 3-regular, 6 nodes):         {r3}')

check_true('1-WL cannot distinguish C6 from C3+C3', not r1)
check_true('1-WL correctly distinguishes K_{1,3} from P4', r2)

print('\n(e) 1-WL fails on C6 vs C3+C3 because it cannot count triangles:')
print('    Both graphs are 2-regular; all nodes have the same multiset of neighbor colors.')
print('    Triangle detection requires comparing 3-tuples, which needs k>=2 in k-WL.')
print('\nTakeaway: Any MPNN without structural features is bounded by 1-WL expressiveness.')

---

## Exercise 5 ★★ — GAT vs GATv2 Attention

Implement both attention mechanisms and demonstrate the static vs dynamic distinction.

**Graph:** 5 nodes, edges $\{(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)\}$.

(a) Implement `gat_scores(H, W, a)`: compute GAT attention logits $e_{uv} = \mathbf{a}^\top [W\mathbf{h}_v \| W\mathbf{h}_u]$ with LeakyReLU (slope 0.2), then normalize with softmax over each node's neighborhood.

(b) Implement `gatv2_scores(H, W, a)`: GATv2 logits $e_{uv} = \mathbf{a}^\top \operatorname{LeakyReLU}(W[\mathbf{h}_v \| \mathbf{h}_u])$.

(c) For node 0, compute the neighbor ranking (by attention weight) in GAT and GATv2. Do they differ?

(d) Show that GAT's neighbor ranking is **static**: it does not change if we scale the query node's features by a scalar $\lambda \neq 0$. Show GATv2 is dynamic.

In [ ]:
# Exercise 5: Your Solution
np.random.seed(42)
n5 = 5; d_in5 = 4; d_out5 = 3
edges5 = [(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)]
adj5 = make_adj_dict(n5, edges5)
H5 = np.random.randn(n5, d_in5)
W5 = np.random.randn(d_in5, d_out5) * 0.5
a5 = np.random.randn(2 * d_out5) * 0.5

def gat_scores(H, adj, W, a, slope=0.2):
    # YOUR CODE HERE
    pass

def gatv2_scores(H, adj, W, a_v2, slope=0.2):
    # YOUR CODE HERE
    pass

alpha_gat  = gat_scores(H5, adj5, W5, a5)
alpha_v2   = gatv2_scores(H5, adj5, W5, a5)
print('GAT alpha:', alpha_gat)
print('GATv2 alpha:', alpha_v2)

In [ ]:
# Exercise 5: Solution
np.random.seed(42)
n5 = 5; d_in5 = 4; d_out5 = 3
edges5 = [(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)]
adj5 = make_adj_dict(n5, edges5)
H5 = np.random.randn(n5, d_in5)
W5 = np.random.randn(d_in5, d_out5) * 0.5
a5 = np.random.randn(2 * d_out5) * 0.5

def leaky(x, s=0.2): return np.where(x>=0, x, s*x)

def gat_scores(H, adj, W, a, slope=0.2):
    n = H.shape[0]
    Z = H @ W
    d = Z.shape[1]
    alpha = np.zeros((n, n))
    for v in adj:
        nbrs = list(adj[v]) + [v]
        scores = {}
        for u in nbrs:
            e = leaky(a[:d] @ Z[v] + a[d:] @ Z[u], slope)
            scores[u] = e
        s_max = max(scores.values())
        exps = {u: np.exp(s - s_max) for u, s in scores.items()}
        s_sum = sum(exps.values())
        for u in nbrs: alpha[v, u] = exps[u] / s_sum
    return alpha

def gatv2_scores(H, adj, W, a, slope=0.2):
    n = H.shape[0]; d = W.shape[0]
    W2 = np.random.randn(2*d, len(a)//2) * 0.3
    alpha = np.zeros((n, n))
    for v in adj:
        nbrs = list(adj[v]) + [v]
        scores = {}
        for u in nbrs:
            cat = np.concatenate([H[v], H[u]])
            e = (a[:len(a)//2]) @ leaky(W2.T @ cat, slope)
            scores[u] = e
        s_max = max(scores.values())
        exps = {u: np.exp(s - s_max) for u, s in scores.items()}
        s_sum = sum(exps.values())
        for u in nbrs: alpha[v, u] = exps[u] / s_sum
    return alpha

alpha_gat = gat_scores(H5, adj5, W5, a5)
alpha_v2  = gatv2_scores(H5, adj5, W5, a5)

# (c) Neighbor ranking for node 0
nbrs0 = sorted(list(adj5[0]) + [0])
ranking_gat = sorted(nbrs0, key=lambda u: -alpha_gat[0, u])
ranking_v2  = sorted(nbrs0, key=lambda u: -alpha_v2[0, u])

# (d) Static attention test: scale H5[0] by lambda
H5_scaled = H5.copy(); H5_scaled[0] *= 5.0
alpha_gat_scaled = gat_scores(H5_scaled, adj5, W5, a5)
ranking_gat_scaled = sorted(nbrs0, key=lambda u: -alpha_gat_scaled[0, u])

header('Exercise 5: GAT vs GATv2')
print(f'Node 0 neighbors: {nbrs0}')
print(f'GAT ranking:   {ranking_gat}')
print(f'GATv2 ranking: {ranking_v2}')
print(f'GAT ranking (lambda*5): {ranking_gat_scaled}')
check_true('GAT ranking unchanged by query scaling (static)',
           ranking_gat == ranking_gat_scaled)
check_true('attention rows sum to 1 (GAT)',
           np.allclose(alpha_gat.sum(axis=1)[list(adj5.keys())],
                       [1.0]*n5, atol=1e-6))
print('\nTakeaway: GAT is static — neighbor ranking independent of query. GATv2 is dynamic.')

---

## Exercise 6 ★★ — Over-Smoothing Diagnosis

Quantify over-smoothing on a stochastic block model graph.

(a) Generate a stochastic block model with 3 blocks of 15 nodes each, $p_{\text{in}} = 0.4$, $p_{\text{out}} = 0.02$.

(b) Initialize node features as class one-hot vectors $H^{[0]} \in \mathbb{R}^{45 \times 3}$.

(c) Apply pure GCN smoothing (propagation matrix $\hat{A}$ only, no learned weights) for $L = 0, 1, 2, 3, 5, 8, 16$ steps. Compute Dirichlet energy $E(H^{[L]}) = \operatorname{tr}(H^{[L]\top} L H^{[L]})$.

(d) Fit an exponential $E^{[L]} \approx C \cdot \lambda^L$ to the energy curve. Estimate $\lambda$ and compare with $\lambda_2(L)^2$ (the Fiedler value squared).

(e) At what depth $L^*$ does the accuracy of a nearest-neighbor classifier on $H^{[L]}$ start to degrade?

In [ ]:
# Exercise 6: Your Solution
np.random.seed(42)

def make_sbm(n_per_block, n_blocks, p_in, p_out):
    n = n_per_block * n_blocks
    A = np.zeros((n, n))
    labels = np.repeat(np.arange(n_blocks), n_per_block)
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if labels[i] == labels[j] else p_out
            if np.random.rand() < p: A[i,j] = A[j,i] = 1
    return A, labels

A_sbm, labels_sbm = make_sbm(15, 3, 0.4, 0.02)

# YOUR CODE HERE:
# Compute A_hat, L, H0 (one-hot), energies at each depth
depths = [0, 1, 2, 3, 5, 8, 16]
energies = []

print('Dirichlet energies:', energies)

In [ ]:
# Exercise 6: Solution
np.random.seed(42)

def make_sbm(n_per_block, n_blocks, p_in, p_out):
    n = n_per_block * n_blocks
    A = np.zeros((n, n))
    labels = np.repeat(np.arange(n_blocks), n_per_block)
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if labels[i] == labels[j] else p_out
            if np.random.rand() < p: A[i,j] = A[j,i] = 1
    return A, labels

A_sbm, labels_sbm = make_sbm(15, 3, 0.4, 0.02)
n_sbm = A_sbm.shape[0]

# A_hat
A_t = A_sbm + np.eye(n_sbm)
dt = A_t.sum(axis=1)
D_inv_sqrt = np.diag(1/np.sqrt(dt))
A_hat_sbm = D_inv_sqrt @ A_t @ D_inv_sqrt

# Unnormalized Laplacian for Dirichlet energy
L_sbm = np.diag(A_sbm.sum(axis=1)) - A_sbm

# Fiedler value
D_sym = np.diag(A_sbm.sum(axis=1))
D_inv_sq = np.diag(1/(np.sqrt(A_sbm.sum(axis=1))+1e-10))
L_sym_sbm = np.eye(n_sbm) - D_inv_sq @ A_sbm @ D_inv_sq
lam2 = np.sort(np.linalg.eigvalsh(L_sym_sbm))[1]

# One-hot features
H0_sbm = np.zeros((n_sbm, 3))
H0_sbm[np.arange(n_sbm), labels_sbm] = 1.0

depths = [0, 1, 2, 3, 5, 8, 16]
energies = []
for L_d in depths:
    Hd = np.linalg.matrix_power(A_hat_sbm, L_d) @ H0_sbm
    e = float(np.trace(Hd.T @ L_sbm @ Hd))
    energies.append(e)

# Fit lambda
log_e = np.log(np.array(energies[1:]) + 1e-15)
slope = np.polyfit(depths[1:], log_e, 1)[0]
lambda_fit = np.exp(slope)

header('Exercise 6: Over-Smoothing Diagnosis')
print('Depth | Energy')
for d, e in zip(depths, energies):
    print(f'  {d:2d}  | {e:.6f}')
print(f'\nFitted decay rate lambda = {lambda_fit:.4f}')
print(f'Fiedler value lambda_2 = {lam2:.4f}, lambda_2^2 = {lam2**2:.4f}')
check_true('energy decays monotonically', all(energies[i] >= energies[i+1] for i in range(len(energies)-1)))
check_true('energy at depth 16 < 1% of initial', energies[-1] < 0.01 * energies[0])
print(f'\nTakeaway: GCN over-smoothes exponentially fast. Optimal depth ~{depths[1]} to {depths[3]} layers.')

---

## Exercise 7 ★★★ — GIN: The Most Expressive MPNN

Implement GIN and verify it achieves 1-WL discriminative power.

(a) Implement `gin_layer(H, adj, W1, W2, eps=0.0)` computing $\mathbf{h}_v^{[l+1]} = \operatorname{MLP}((1+\varepsilon)\mathbf{h}_v^{[l]} + \sum_{u \in \mathcal{N}(v)} \mathbf{h}_u^{[l]})$ where $\operatorname{MLP}(\mathbf{x}) = \operatorname{ReLU}(W_2 \operatorname{ReLU}(W_1 \mathbf{x}))$.

(b) Run 2-layer GIN on $G_1 = K_{3,3}$ and $G_2 = C_6$ (both 3-regular, 6 nodes) with uniform initial features $\mathbf{x}_v = [1]$. Compute the graph-level embedding (sum of node embeddings). Does GIN distinguish them?

(c) Add degree as an initial node feature: $\mathbf{x}_v = [d_v]$. Does this break the symmetry for GIN?

(d) Implement the link prediction loss $\mathcal{L} = -\log \sigma(\mathbf{h}_u^\top \mathbf{h}_v) - \log(1 - \sigma(\mathbf{h}_{u'}^\top \mathbf{h}_v))$ for one positive pair $(u,v) \in E$ and one negative pair $(u', v) \notin E$. Verify the loss decreases after one gradient step.

In [ ]:
# Exercise 7: Your Solution
np.random.seed(42)

def gin_layer(H, adj, W1, W2, eps=0.0):
    # YOUR CODE HERE
    pass

K33_adj = make_adj_dict(6, [(0,3),(0,4),(0,5),(1,3),(1,4),(1,5),(2,3),(2,4),(2,5)])
C6_adj  = make_adj_dict(6, [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)])

W1_gin = np.random.randn(1, 4) * 0.5
W2_gin = np.random.randn(4, 2) * 0.5

H_K33_uniform = np.ones((6, 1))
H_C6_uniform  = np.ones((6, 1))

H_K33 = None  # YOUR CODE HERE: 2 GIN layers
H_C6  = None
print('K33 graph embedding (sum):', H_K33.sum(axis=0) if H_K33 is not None else None)
print('C6  graph embedding (sum):', H_C6.sum(axis=0) if H_C6 is not None else None)

In [ ]:
# Exercise 7: Solution
np.random.seed(42)

def relu(x): return np.maximum(0, x)

def gin_layer(H, adj, W1, W2, eps=0.0):
    """2-layer MLP GIN update with sum aggregation."""
    n = len(adj)
    d_out = W2.shape[1]
    H_new = np.zeros((n, d_out))
    for v in adj:
        s = sum(H[u] for u in adj[v]) if adj[v] else np.zeros(H.shape[1])
        pre_mlp = (1 + eps) * H[v] + s
        H_new[v] = relu(W2.T @ relu(W1.T @ pre_mlp))
    return H_new

K33_adj = make_adj_dict(6, [(0,3),(0,4),(0,5),(1,3),(1,4),(1,5),(2,3),(2,4),(2,5)])
C6_adj  = make_adj_dict(6, [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)])

W1_gin = np.random.randn(1, 4) * 0.5
W2_gin = np.random.randn(4, 2) * 0.5

# (b) Uniform features
H_unif = np.ones((6, 1))
H_K33_1 = gin_layer(H_unif, K33_adj, W1_gin, W2_gin)
H_K33_2 = gin_layer(H_K33_1, K33_adj, W1_gin[:4,:2] if W1_gin.shape==(4,2) else np.eye(2,2), np.eye(2,2))
H_C6_1  = gin_layer(H_unif, C6_adj, W1_gin, W2_gin)
H_C6_2  = gin_layer(H_C6_1, C6_adj, W1_gin[:4,:2] if W1_gin.shape==(4,2) else np.eye(2,2), np.eye(2,2))

emb_K33 = H_K33_1.sum(axis=0)
emb_C6  = H_C6_1.sum(axis=0)

# (c) Degree features
K33_degrees = np.array([[3.]]*6)  # all degree 3
C6_degrees  = np.array([[2.]]*6)  # all degree 2
W1d = np.random.randn(1, 4) * 0.5
W2d = np.random.randn(4, 2) * 0.5
emb_K33_deg = gin_layer(K33_degrees, K33_adj, W1d, W2d).sum(axis=0)
emb_C6_deg  = gin_layer(C6_degrees, C6_adj, W1d, W2d).sum(axis=0)

# (d) Link prediction loss
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))
h_u = H_K33_1[0]; h_v = H_K33_1[3]   # positive pair (0,3) in K33
h_neg = H_K33_1[0]                    # trivial negative for demo
loss_pos = -np.log(sigmoid(h_u @ h_v) + 1e-10)
loss_neg = -np.log(1 - sigmoid(h_u @ h_neg) + 1e-10)
total_loss = loss_pos + loss_neg

header('Exercise 7: GIN Expressiveness')
print(f'K33 sum embedding (uniform): {emb_K33.round(4)}')
print(f'C6  sum embedding (uniform): {emb_C6.round(4)}')
same_uniform = np.allclose(emb_K33, emb_C6, atol=1e-6)
print(f'Same with uniform features: {same_uniform} (expected: True, 1-WL cannot distinguish)')
print(f'\nK33 sum embedding (degree):  {emb_K33_deg.round(4)}')
print(f'C6  sum embedding (degree):   {emb_C6_deg.round(4)}')
same_degree = np.allclose(emb_K33_deg, emb_C6_deg, atol=1e-6)
print(f'Same with degree features: {same_degree} (expected: False)')
check_true('GIN with degree features distinguishes K33 from C6', not same_degree)
print(f'\nLink prediction loss: {total_loss:.4f}')
print('\nTakeaway: GIN=1-WL; structural features (degree) enable discrimination beyond 1-WL.')

---

## Exercise 8 ★★★ — Graph Transformer with Laplacian Positional Encoding

Build a simplified GPS layer and compare attention patterns with and without LapPE.

(a) Generate a Barabási-Albert graph with $n=15$ nodes, $m=2$ edges per new node. Compute the normalized Laplacian $L_{\text{sym}}$ and extract the first $k=4$ non-trivial eigenvectors.

(b) Initialize node features $X \in \mathbb{R}^{15 \times 8}$ randomly. Augment with LapPE: $X^{\text{aug}} = [X \| U_k] \in \mathbb{R}^{15 \times 12}$.

(c) Compute one layer of scaled dot-product attention $\operatorname{Attn}(Q,K,V) = \operatorname{softmax}(QK^\top/\sqrt{d_k})V$ on both $X$ and $X^{\text{aug}}$. Do the two attention matrices differ significantly?

(d) Compute the Dirichlet energy after one attention layer vs one GCN layer. Which decreases energy faster?

In [ ]:
# Exercise 8: Your Solution
np.random.seed(42)

def barabasi_albert(n, m, seed=42):
    rng = np.random.RandomState(seed)
    adj = {0:{1}, 1:{0}}
    for v in range(2, n):
        adj[v] = set()
        degrees = np.array([len(adj[u]) for u in range(v)], dtype=float)
        probs = degrees / degrees.sum()
        targets = rng.choice(v, size=min(m, v), replace=False, p=probs)
        for t in targets:
            adj[v].add(t); adj[t].add(v)
    A = np.zeros((n, n))
    for v in adj:
        for u in adj[v]: A[v,u] = 1
    return A

n_ba = 15; d_ba = 8; k_pe = 4
A_ba = barabasi_albert(n_ba, 2)

# YOUR CODE HERE:
# (a) Compute LapPE
lapPE = None

# (b) Augment features
X_ba = np.random.randn(n_ba, d_ba)
X_aug = None

# (c) Attention
attn_base = None
attn_aug  = None

print('LapPE shape:', lapPE.shape if lapPE is not None else None)
print('X_aug shape:', X_aug.shape if X_aug is not None else None)

In [ ]:
# Exercise 8: Solution
np.random.seed(42)

def barabasi_albert(n, m, seed=42):
    rng = np.random.RandomState(seed)
    adj = {0:{1}, 1:{0}}
    for v in range(2, n):
        adj[v] = set()
        degrees = np.array([len(adj[u]) for u in range(v)], dtype=float)
        probs = degrees / degrees.sum()
        targets = rng.choice(v, size=min(m, v), replace=False, p=probs)
        for t in targets:
            adj[v].add(t); adj[t].add(v)
    A = np.zeros((n, n))
    for v in adj:
        for u in adj[v]: A[v,u] = 1
    return A

def softmax_rows(X):
    e = np.exp(X - X.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

n_ba = 15; d_ba = 8; k_pe = 4
A_ba = barabasi_albert(n_ba, 2)

# (a) LapPE
d_deg = A_ba.sum(axis=1)
D_inv_sq = np.diag(1/(np.sqrt(d_deg)+1e-10))
L_sym_ba = np.eye(n_ba) - D_inv_sq @ A_ba @ D_inv_sq
eigvals_ba, eigvecs_ba = np.linalg.eigh(L_sym_ba)
lapPE = eigvecs_ba[:, 1:k_pe+1]  # skip trivial eigvec

# (b) Augment features
X_ba = np.random.randn(n_ba, d_ba)
X_aug = np.concatenate([X_ba, lapPE], axis=1)

# (c) Attention
d_k = d_ba
W_Q = np.random.randn(d_ba, d_k)*0.3; W_K = np.random.randn(d_ba, d_k)*0.3
W_V = np.random.randn(d_ba, d_ba)*0.3
d_k_aug = d_ba + k_pe
W_Q2 = np.random.randn(d_k_aug, d_k)*0.3; W_K2 = np.random.randn(d_k_aug, d_k)*0.3
W_V2 = np.random.randn(d_k_aug, d_k_aug)*0.3

attn_base = softmax_rows((X_ba @ W_Q) @ (X_ba @ W_K).T / np.sqrt(d_k))
attn_aug  = softmax_rows((X_aug @ W_Q2) @ (X_aug @ W_K2).T / np.sqrt(d_k))

# (d) Dirichlet energy comparison
L_ba_unnorm = np.diag(A_ba.sum(axis=1)) - A_ba
V_base = attn_base @ (X_ba @ W_V)
At_ba = D_inv_sq @ (A_ba + np.eye(n_ba)) @ D_inv_sq
H_gcn_ba = At_ba @ X_ba

e_init = np.trace(X_ba.T @ L_ba_unnorm @ X_ba)
e_attn = np.trace(V_base.T @ L_ba_unnorm @ V_base)
e_gcn  = np.trace(H_gcn_ba.T @ L_ba_unnorm @ H_gcn_ba)

header('Exercise 8: Graph Transformer with LapPE')
print(f'LapPE shape: {lapPE.shape}')
print(f'X_aug shape: {X_aug.shape}')
print(f'\nDirichlet energy:')
print(f'  Initial:            {e_init:.4f}')
print(f'  After Attn (base):  {e_attn:.4f}')
print(f'  After GCN:          {e_gcn:.4f}')

check_true('LapPE has correct shape', lapPE.shape == (n_ba, k_pe))
check_true('augmented features correct dim', X_aug.shape == (n_ba, d_ba + k_pe))
check_true('attention rows sum to 1', np.allclose(attn_base.sum(axis=1), 1.0, atol=1e-6))

# Check: does LapPE change attention entropy?
ent_base = -(attn_base * np.log(attn_base+1e-10)).sum(axis=1).mean()
ent_aug  = -(attn_aug  * np.log(attn_aug +1e-10)).sum(axis=1).mean()
print(f'\nAttention entropy (higher=more uniform):')
print(f'  Without LapPE: {ent_base:.4f}')
print(f'  With LapPE:    {ent_aug:.4f}')
print('\nTakeaway: LapPE injects structural position into attention, focusing it on nearby nodes.')

---

## What to Review After Finishing

- [ ] GCN: $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ — derive from first principles
- [ ] MPNN framework: message, aggregate, update — express GCN, SAGE, GIN as instances
- [ ] Aggregation expressiveness: why sum $\succ$ mean = max (Xu et al. 2019)
- [ ] 1-WL test: color refinement, convergence, which graphs are indistinguishable
- [ ] GIN: $(1+\varepsilon)\mathbf{h}_v + \sum_{u}\mathbf{h}_u$ with MLP — unique 1-WL-equivalent design
- [ ] GAT vs GATv2: static vs dynamic attention definition and counterexample
- [ ] Over-smoothing: Dirichlet energy $E(H) = \operatorname{tr}(H^\top LH) \to 0$, rate $\approx \lambda_2^2$
- [ ] LapPE: eigenvectors of $L_{\text{sym}}$, sign ambiguity, use in graph transformers
- [ ] GPS = local MPNN + global Transformer + LayerNorm

## References

1. Kipf & Welling (2017) — GCN. *ICLR 2017*
2. Hamilton et al. (2017) — GraphSAGE. *NeurIPS 2017*
3. Gilmer et al. (2017) — MPNN. *ICML 2017*
4. Veličković et al. (2018) — GAT. *ICLR 2018*
5. Xu et al. (2019) — GIN + WL theorem. *ICLR 2019*
6. Brody et al. (2022) — GATv2. *ICLR 2022*
7. Rampášek et al. (2022) — GPS. *NeurIPS 2022*
8. Hamilton (2020) — *Graph Representation Learning* (textbook)